In [51]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, losses

In [52]:
MAX_VOCAB = 1000
CONTEXT_WIN = 20
EMBED_WIN = 64
HEADS = 2
FEED_FOWARD = 64
TRANSFORMER_BLOCKS = 2
BATCH_SIZE = 16
EPOCHS = 100

train_text = [
"Machine Learning is very hard",
"Touch my ass again and imma smack you",
"I need a job",
"GitHub is goated",
"Saber Class really is made out of Sabers",
"Skibidi Toilet Green Giant"
]

In [53]:
tokenizer = layers.TextVectorization(
    max_tokens=MAX_VOCAB,
    output_mode="int",
    output_sequence_length=CONTEXT_WIN
)

tokenizer.adapt(train_text)
vocab = tokenizer.get_vocabulary()

#next token prediction
def prep_data(train_text):
  seq = tokenizer(train_text)
  x_input = seq[:, :-1]
  y_target = seq[:, 1:]

  return x_input, y_target

x_train, y_train = prep_data(train_text)

In [54]:
def perplexity(true,pred):
  return tf.exp(tf.reduce_mean(losses.sparse_categorical_crossentropy(true,pred)))

class TokenPositionEmbedding(layers.Layer):
  def __init__(self, context_win, max_vocab, embed_dim, **kwargs) -> None:
    super().__init__()
    self.token_embed = layers.Embedding(input_dim = max_vocab, output_dim=embed_dim)
    self.position_embed = layers.Embedding(input_dim = context_win, output_dim=embed_dim)

  def call(self, x):
    context_win = tf.shape(x)[-1]
    positions = self.position_embed(tf.range(start=0, limit=context_win, delta=1))
    return self.token_embed(x) + positions


Each head computes:

attention(Q,K,V) = softmax(QK/sqrt(d_k))*V

basically word context

transformer block:

Input

   │

   ├── Self Attention

   ├── Add (Residual)

   ├── LayerNorm

   │

   ├── Feed Forward

   ├── Add (Residual)

   ├── LayerNorm

   │

Output



In [55]:
class TransformerBlock(layers.Layer):
  def __init__(self, embed_dim, heads, feed_forward, **kwargs) -> None:
    super().__init__()
    self.attention = layers.MultiHeadAttention(num_heads=heads, key_dim=embed_dim)
    self.feed_foward_net = models.Sequential([layers.Dense(feed_forward, activation="relu"), layers.Dense(embed_dim)])
    self.norm1 = layers.LayerNormalization(epsilon=1e-4)
    self.norm2 = layers.LayerNormalization(epsilon=1e-4)
    self.drop1 = layers.Dropout(0.1)
    self.drop2 = layers.Dropout(0.1)

  def call(self, inputs, training = False):
    #Self Attention
    attention_output = self.attention(inputs, inputs, use_causal_mask=True)
    attention_output = self.drop1(attention_output, training=training)
    output = self.norm1(inputs + attention_output)

    #Feed Forward Network
    feed_forward_output = self.feed_foward_net(output)
    feed_foward_output = self.drop2(feed_forward_output, training=training)

    return self.norm2(output + feed_forward_output)

In [56]:
class LLM(models.Model):
  def __init__(self, context_win, max_vocab, embed_dim, heads, feed_forward, blocks, **kwargs) -> None:
    super().__init__(**kwargs)
    self.embed_layer = TokenPositionEmbedding(context_win, max_vocab, embed_dim)
    self.transformer_blocks = [TransformerBlock(embed_dim, heads, feed_forward) for _ in range(blocks)]
    self.dense_out = layers.Dense(max_vocab)

  def call(self, inputs, training=False):
      x = self.embed_layer(inputs)
      for block in self.transformer_blocks:
        x = block(x, training=training)
      return self.dense_out(x)


  def gen(model, prompt, length = 10, temperature= 1.0, top_k = 5):
    input_tensor = tokenizer([prompt])
    tokens = [token for token in input_tensor.numpy()[0] if token != 0]

    gen_text = prompt
    for _ in range(length):
      context_token = tokens[-CONTEXT_WIN:]
      input_data = tf.convert_to_tensor([context_token])

      preds = model(input_data, training=False)
      next_logits = preds[0, -1, :]
      next_logits /= (temperature + 1e-7)
      top_val,  top_idx = tf.math.top_k(next_logits, k=top_k)
      top_prob = tf.nn.softmax(top_val).numpy()

      next_idx = np.random.choice(top_idx.numpy(), p=top_prob)
      if next_idx == 0 and len(tokens) >0:
        next_idx = top_idx.numpy()[1]

      tokens.append(next_idx)
      gen_text += " " + vocab[next_idx]

    return gen_text

In [57]:
skibidi_model = LLM(CONTEXT_WIN, len(vocab), EMBED_WIN, HEADS, FEED_FOWARD, TRANSFORMER_BLOCKS)
skibidi_model.compile(optimizer="adam", loss=losses.SparseCategoricalCrossentropy(from_logits=True), metrics=[perplexity])
skibidi_model.fit(x_train, y_train, batch_size=BATCH_SIZE, epochs=EPOCHS, verbose=1)

Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 16s 16s/step - loss: 3.3300 - perplexity: 328.6434
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - loss: 1.7205 - perplexity: 50.6231
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - loss: 1.1779 - perplexity: 36.4381
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - loss: 1.0111 - perplexity: 25.5760
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - loss: 0.9149 - perplexity: 20.4910
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step - loss: 0.8381 - perplexity: 13.0032
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - loss: 0.7515 - perplexity: 10.3401
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step - loss: 0.6581 - perplexity: 9.3461
Epoch 9/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - loss: 0.5748 - perplexity: 8.9067
Epoch 10/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - loss: 0.5052 - perplexity: 7.8515
Epoch 11/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - loss: 0.4346 - perplexity: 7.4150
Epoch 12/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 5

In [61]:
test_input1 = input("Enter a prompt: ")
test_input2 = input("Enter a prompt: ")
print(skibidi_model.gen(test_input1))
print(skibidi_model.gen(test_input2))

Enter a prompt: is
Enter a prompt: made
is goated goated again and imma smack you made out of
made out of giant again imma smack you made out of
